# Tahap 5 — Training & Testing the Model
Hanya data **train 2024** yang dipakai. Test dan OOT tidak dibaca di notebook ini.

In [1]:
import os
import platform

import imblearn
import joblib
import pandas as pd
import sklearn
import xgboost
from sklearn.base import clone

from src import config
from src.data_prep import load_processed
from src.reporting import fmt_auto, fmt_id, save_table
from src.train import MODEL_NAMES, SCORING, cv_evaluate, make_model_specs, make_pipeline, oof_proba, tune

X_train, y_train, _, _ = load_processed("train")
pos_weight = float((y_train == 0).sum() / (y_train == 1).sum())
print(X_train.shape, f"positif {y_train.mean():.2%}", f"pos_weight {pos_weight:.3f}")

(44001, 13) positif 11.75% pos_weight 7.509


## 1. Tuning, CV, dan OOF predictions per model

In [2]:
results, cv_tables, oof = {}, [], pd.DataFrame({"y": y_train}, index=X_train.index)
for name in MODEL_NAMES:
    r = tune(name, X_train, y_train)
    cv = cv_evaluate(r.pipeline, X_train, y_train)
    cv.insert(0, "model", name)
    cv_tables.append(cv)
    oof[name] = oof_proba(r.pipeline, X_train, y_train)
    joblib.dump(r.pipeline, config.MODELS / f"{name}.joblib")
    results[name] = r
    print(f"{name}: {r.fit_seconds:.1f} s | best {r.best_params}")
    print(cv.round(4).to_string(index=False))

oof.to_parquet(config.DATA_PROCESSED / "oof_proba_train.parquet")

dummy: 0.1 s | best {}
model            metrik   mean    std
dummy average_precision 0.1172 0.0007
dummy           roc_auc 0.4983 0.0035
dummy            recall 0.1102 0.0062
dummy         precision 0.1145 0.0065
dummy                f1 0.1123 0.0063


logreg: 14.4 s | best {'clf__C': np.float64(0.006026889128682512)}
 model            metrik   mean    std
logreg average_precision 0.6589 0.0080
logreg           roc_auc 0.8656 0.0039
logreg            recall 0.7513 0.0127
logreg         precision 0.4486 0.0109
logreg                f1 0.5617 0.0113


rf: 356.8 s | best {'clf__max_depth': 14, 'clf__max_features': 0.5, 'clf__min_samples_leaf': 11, 'clf__n_estimators': 287}
model            metrik   mean    std
   rf average_precision 0.7925 0.0045
   rf           roc_auc 0.9055 0.0033
   rf            recall 0.7463 0.0087
   rf         precision 0.7229 0.0117
   rf                f1 0.7343 0.0072


xgb: 58.2 s | best {'clf__colsample_bytree': np.float64(0.8736932106048627), 'clf__learning_rate': np.float64(0.03738105868191797), 'clf__max_depth': 6, 'clf__min_child_weight': 8, 'clf__n_estimators': 234, 'clf__subsample': np.float64(0.9637281608315128)}
model            metrik   mean    std
  xgb average_precision 0.7963 0.0052
  xgb           roc_auc 0.9072 0.0030
  xgb            recall 0.7662 0.0069
  xgb         precision 0.6690 0.0137
  xgb                f1 0.7142 0.0084


In [3]:
cv_all = pd.concat(cv_tables, ignore_index=True)
cv_all["mean_std"] = [f"{fmt_id(m, 3)} ± {fmt_id(s, 3)}" for m, s in zip(cv_all["mean"], cv_all["std"])]
save_table(cv_all, "t5_cv_results", decimals=4)
cv_all.pivot(index="model", columns="metrik", values="mean_std").loc[MODEL_NAMES]

metrik,average_precision,f1,precision,recall,roc_auc
model,,,,,
dummy,"0,117 ± 0,001","0,112 ± 0,006","0,114 ± 0,006","0,110 ± 0,006","0,498 ± 0,004"
logreg,"0,659 ± 0,008","0,562 ± 0,011","0,449 ± 0,011","0,751 ± 0,013","0,866 ± 0,004"
rf,"0,792 ± 0,005","0,734 ± 0,007","0,723 ± 0,012","0,746 ± 0,009","0,906 ± 0,003"
xgb,"0,796 ± 0,005","0,714 ± 0,008","0,669 ± 0,014","0,766 ± 0,007","0,907 ± 0,003"


In [4]:
best_rows = []
for name, r in results.items():
    params = {k.replace("clf__", ""): v for k, v in r.best_params.items()}
    best_rows.append({
        "model": name,
        "best_params": ", ".join(f"{k}={fmt_id(v, 4) if isinstance(v, float) else v}" for k, v in params.items()) or "-",
        "waktu_tuning_detik": r.fit_seconds,
        "n_data_tuning": r.tuned_on_n,
    })
best = pd.DataFrame(best_rows)
save_table(best, "t5_best_params", decimals=1)
best

,model,best_params,waktu_tuning_detik,n_data_tuning
0,dummy,-,0.120974,44001
1,logreg,"C=0,0060",14.419359,44001
2,rf,"max_depth=14, max_features=0,5000, min_samples...",356.758897,44001
3,xgb,"colsample_bytree=0,8737, learning_rate=0,0374,...",58.189928,44001


## 2. Eksperimen imbalance pada model terbaik (CV PR-AUC)

In [5]:
ap = cv_all[cv_all["metrik"] == "average_precision"].set_index("model")["mean"]
best_name = ap.drop("dummy").idxmax()
print("Model terbaik menurut CV PR-AUC:", best_name)

weighted = results[best_name].pipeline
base_clf = clone(weighted.named_steps["clf"])
if best_name == "xgb":
    base_clf.set_params(scale_pos_weight=1)
else:
    base_clf.set_params(class_weight=None)
smote_pipe = make_pipeline(base_clf, smote=True)

imb_rows = []
for label, pipe in (("class weighting", weighted), ("SMOTE (tanpa bobot kelas)", smote_pipe)):
    t = cv_evaluate(pipe, X_train, y_train)
    t.insert(0, "strategi", label)
    imb_rows.append(t)
imb = pd.concat(imb_rows, ignore_index=True)
imb.insert(0, "model", best_name)
imb["mean_std"] = [f"{fmt_id(m, 3)} ± {fmt_id(s, 3)}" for m, s in zip(imb["mean"], imb["std"])]
save_table(imb, "t5_imbalance_experiment", decimals=4)
imb.pivot(index="strategi", columns="metrik", values="mean_std")

Model terbaik menurut CV PR-AUC: xgb


metrik,average_precision,f1,precision,recall,roc_auc
strategi,,,,,
SMOTE (tanpa bobot kelas),"0,790 ± 0,006","0,741 ± 0,006","0,755 ± 0,011","0,727 ± 0,008","0,902 ± 0,004"
class weighting,"0,796 ± 0,005","0,714 ± 0,008","0,669 ± 0,014","0,766 ± 0,007","0,907 ± 0,003"


## 3. Setup eksperimen dan ruang hyperparameter

In [6]:
mem_gb = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1024**3
cpu = next((l.split(":")[1].strip() for l in open("/proc/cpuinfo") if l.startswith("model name")), platform.processor())
setup = pd.DataFrame([
    ("Data latih", f"{fmt_id(len(X_train))} baris (2024, 80% stratified), {X_train.shape[1]} fitur input"),
    ("Proporsi kelas positif (ditolak)", f"{fmt_id(y_train.mean() * 100, 2)}% (rasio negatif:positif {fmt_id(pos_weight, 2)})"),
    ("Validasi", f"StratifiedKFold {config.CV_FOLDS} fold, shuffle, random_state {config.RANDOM_STATE}"),
    ("Pencarian hyperparameter", f"RandomizedSearchCV, n_iter {config.N_ITER}, refit pada PR-AUC (average_precision)"),
    ("Metrik CV", ", ".join(SCORING)),
    ("Subsample tuning", f"Tidak (data latih {fmt_id(len(X_train))} ≤ ambang {fmt_id(config.TUNE_SUBSAMPLE_TRIGGER)} baris)"),
    ("Penanganan imbalance", "Utama: class weighting (balanced / scale_pos_weight); pembanding: SMOTE di dalam imblearn Pipeline"),
    ("Preprocessing", "Di dalam Pipeline: capping persentil 1–99, log1p (3 fitur nominal), imputasi median (tanpa missing indicator), StandardScaler, one-hot"),
    ("Seed", f"RANDOM_STATE = {config.RANDOM_STATE} di semua komponen"),
    ("Library", f"Python {platform.python_version()}, scikit-learn {sklearn.__version__}, xgboost {xgboost.__version__}, imbalanced-learn {imblearn.__version__}, pandas {pd.__version__}"),
    ("Perangkat keras", f"{cpu}, {os.cpu_count()} thread, RAM {fmt_id(mem_gb, 1)} GB; n_jobs = {config.N_JOBS}"),
    ("Test & OOT", "Tidak dipakai di tahap ini (disimpan untuk Tahap 6–7)"),
], columns=["aspek", "keterangan"])
save_table(setup, "t5_experiment_setup")
setup

,aspek,keterangan
0,Data latih,"44.001 baris (2024, 80% stratified), 13 fitur ..."
1,Proporsi kelas positif (ditolak),"11,75% (rasio negatif:positif 7,51)"
2,Validasi,"StratifiedKFold 5 fold, shuffle, random_state 42"
3,Pencarian hyperparameter,"RandomizedSearchCV, n_iter 25, refit pada PR-A..."
4,Metrik CV,"average_precision, roc_auc, recall, precision, f1"
5,Subsample tuning,Tidak (data latih 44.001 ≤ ambang 200.000 baris)
6,Penanganan imbalance,Utama: class weighting (balanced / scale_pos_w...
7,Preprocessing,"Di dalam Pipeline: capping persentil 1–99, log..."
8,Seed,RANDOM_STATE = 42 di semua komponen
9,Library,"Python 3.11.18, scikit-learn 1.9.1, xgboost 3...."


In [7]:
REASON = {
    "C": "Kekuatan regularisasi L2 (C kecil = regularisasi kuat); meredam multikolinearitas fitur ukuran",
    "n_estimators": "Jumlah pohon; lebih banyak = lebih stabil tetapi lebih lambat",
    "max_depth": "Kedalaman pohon; mengendalikan kompleksitas dan overfitting",
    "min_samples_leaf": "Minimal sampel per daun; regularisasi pohon pada kelas minoritas",
    "max_features": "Proporsi fitur per split; menambah keragaman pohon",
    "learning_rate": "Langkah boosting; kecil = lebih general tetapi butuh lebih banyak pohon",
    "subsample": "Proporsi baris per pohon; mengurangi overfitting",
    "colsample_bytree": "Proporsi fitur per pohon; mengurangi overfitting",
    "min_child_weight": "Minimal bobot per daun; regularisasi",
}
space_rows = []
for name, (est, space) in make_model_specs(pos_weight).items():
    if not space:
        space_rows.append({"model": name, "hyperparameter": "strategy", "ruang_pencarian": "stratified (tetap)", "alasan": "Baseline acak sesuai proporsi kelas"})
    for k, dist in space.items():
        p = k.replace("clf__", "")
        if isinstance(dist, list):
            rng = ", ".join(fmt_id(v, 1) if isinstance(v, float) else str(v) for v in dist)
        elif dist.dist.name == "randint":
            lo, hi = dist.support()
            rng = f"integer {lo}–{hi}"
        elif dist.dist.name == "loguniform":
            rng = f"log-uniform {fmt_auto(dist.args[0])}–{fmt_auto(dist.args[1])}"
        else:
            rng = f"uniform {fmt_id(dist.args[0], 1)}–{fmt_id(dist.args[0] + dist.args[1], 1)}"
        space_rows.append({"model": name, "hyperparameter": p, "ruang_pencarian": rng, "alasan": REASON[p]})
fixed = {
    "logreg": "class_weight=balanced, max_iter=2000, penalty L2",
    "rf": "class_weight=balanced_subsample",
    "xgb": f"scale_pos_weight={fmt_id(pos_weight, 3)} (rasio negatif/positif), tree_method=hist, eval_metric=aucpr",
}
for name, txt in fixed.items():
    space_rows.append({"model": name, "hyperparameter": "(tetap)", "ruang_pencarian": txt, "alasan": "Penanganan imbalance / konfigurasi dasar"})
space_tbl = pd.DataFrame(space_rows)
save_table(space_tbl, "t5_hyperparameter_space")
space_tbl

,model,hyperparameter,ruang_pencarian,alasan
0,dummy,strategy,stratified (tetap),Baseline acak sesuai proporsi kelas
1,logreg,C,"log-uniform 0,001–100",Kekuatan regularisasi L2 (C kecil = regularisa...
2,rf,n_estimators,integer 200–500,Jumlah pohon; lebih banyak = lebih stabil teta...
3,rf,max_depth,"6, 10, 14, 20, None",Kedalaman pohon; mengendalikan kompleksitas da...
4,rf,min_samples_leaf,integer 1–50,Minimal sampel per daun; regularisasi pohon pa...
5,rf,max_features,"sqrt, 0,3, 0,5",Proporsi fitur per split; menambah keragaman p...
6,xgb,n_estimators,integer 200–800,Jumlah pohon; lebih banyak = lebih stabil teta...
7,xgb,max_depth,integer 3–8,Kedalaman pohon; mengendalikan kompleksitas da...
8,xgb,learning_rate,"log-uniform 0,01–0,2",Langkah boosting; kecil = lebih general tetapi...
9,xgb,subsample,"uniform 0,6–1,0",Proporsi baris per pohon; mengurangi overfitting
